In [ ]:
import pandas as pd
import re
import nltk

# nltk.download('punkt_tab')
# nltk.download('stopwords', quiet=True)
# nltk.download('wordnet', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

In [ ]:
data = pd.read_csv("secret_topics_dataset.csv")
data.head()

,doc_id,text
0,DOC_0001,To: ad994@Freenet.carleton.ca JW> 1) Where is...
1,DOC_0002,Hello netters! I have a fairly weak question...
2,DOC_0003,I would probobly feel a hell of a lot poorer.....
3,DOC_0004,One thing that everyone is forgetting in this ...
4,DOC_0005,This is a two-sided problem. Unfortunately our...


In [ ]:
data.isnull().sum()

,0
doc_id,0
text,0


In [ ]:
# initialize lemmatizer and load stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# I used a function, because I'll use the same preprocessing on every message I have
def preprocessing(text):
    if pd.isna(text):
      return ""
    # remove punctuation and marks using regex
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\d+', '', text) # I don't if I should remove numbers or not

    # convert to lowercase (normalization)
    text = text.lower()

    # tokenize the text using word tokenizer
    words = nltk.word_tokenize(text)

    # remove Stop words
    words = [word for word in words if word not in stop_words]

    # use lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    # Join the words back into a single string
    return " ".join(words)

In [ ]:
data['clean_text'] = data['text'].apply(preprocessing)
data['clean_text'].head()

,clean_text
0,adfreenetcarletonca jw book code teach jw read...
1,hello netters fairly weak question ask everybo...
2,would probobly feel hell lot poorer gas tank j...
3,one thing everyone forgetting argument pricing...
4,twosided problem unfortunately culture deterio...


In [ ]:
data['doc_id'] = data['doc_id'].str.replace('DOC_', '', regex=False).astype(int)
data.head()

,doc_id,text,clean_text
0,1,To: ad994@Freenet.carleton.ca JW> 1) Where is...,adfreenetcarletonca jw book code teach jw read...
1,2,Hello netters! I have a fairly weak question...,hello netters fairly weak question ask everybo...
2,3,I would probobly feel a hell of a lot poorer.....,would probobly feel hell lot poorer gas tank j...
3,4,One thing that everyone is forgetting in this ...,one thing everyone forgetting argument pricing...
4,5,This is a two-sided problem. Unfortunately our...,twosided problem unfortunately culture deterio...


In [ ]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(data['clean_text'])
y = data['doc_id']

print("TF-IDF Shape:", X.shape)

TF-IDF Shape: (1070, 13852)


In [ ]:
lsa = TruncatedSVD(n_components=2, random_state=42)
X_lsa = lsa.fit_transform(X)
print("LSA Shape:", X_lsa.shape)

LSA Shape: (1070, 2)


In [ ]:
terms = vectorizer.get_feature_names_out()

n_top_words = 10

for i, component in enumerate(lsa.components_):
    top_words = [terms[index] for index in component.argsort()[:-n_top_words-1:-1]]

    print(f"\nTopic {i+1}")
    print(", ".join(top_words))


Topic 1
car, would, file, know, like, one, image, get, dont, thanks

Topic 2
car, dealer, engine, price, year, mile, new, speed, oil, one
